# Seminar 7: Gradient Boosting for Ranking

## Goals

In this seminar we will:
1. Build a full **learning-to-rank (LTR) pipeline** on the **Yambda** music streaming dataset
2. Leverage **multiple event types** (listens, completion rates, likes, dislikes) for richer features
3. Generate **candidate tracks** using popularity and ALS-based retrieval
4. Engineer **ranking features** from all event types: item quality, user taste, user-item history
5. Understand and implement three families of ranking losses: **pointwise**, **pairwise**, **listwise**
6. For pairwise: train and compare **PairLogit** vs **YetiRank** in CatBoost
6. Train all four **CatBoost ranking losses** (Logloss, PairLogit, YetiRank, QuerySoftMax) and compare them head-to-head
8. Study how **negative sampling strategies** affect ranking quality
9. Evaluate all models with **NDCG@10**, **Hit Rate@10**, and **MRR@10**

---

## Why Multiple Event Types Matter

Real streaming services observe a much richer interaction log:

| Signal | Type | Meaning for ranking |
|--------|------|---------------------|
| `listen` | Implicit | User was exposed and didn't skip |
| `played_ratio_pct` | Implicit | How much they actually engaged (0–100%+) |
| `is_organic` | Context | User discovered this themselves vs. via recommendation |
| `like` | Explicit | Strong positive signal |
| `dislike` | Explicit | Strong negative signal |
| `unlike` / `undislike` | Correction | User changed their mind |

This gives us richer **labels** (graded relevance from completion + likes) and more informative **features** (per-item like rate, per-user taste profile).

---

## The Two-Stage Recommendation Architecture

```
All tracks (millions)
      │
      ▼
  [Stage 1: Retrieval / Candidate Generation]
  Fast: ALS, popularity, BM25, audio similarity
      │
      ▼
Candidates (~100–500 tracks per user)
      │
      ▼
  [Stage 2: Ranking]   ← This seminar
  Gradient Boosting with rich features
      │
      ▼
Final ranked list (top-10 played to user)
```

In [ ]:
!pip install implicit catboost lightgbm xgboost datasets tqdm --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import implicit
from catboost import CatBoostClassifier, CatBoostRanker, Pool
import lightgbm as lgb
import xgboost as xgb

plt.style.use('ggplot')
np.random.seed(42)

print('implicit  :', implicit.__version__)
print('catboost  :', __import__('catboost').__version__)
print('lightgbm  :', lgb.__version__)
print('xgboost   :', xgb.__version__)

---
## Part 1: Dataset — Yambda-50M

**Yambda** is Yandex Music's open large-scale recommendation dataset. We use the **50M** version:

| Split | Users | Tracks | Events |
|-------|-------|--------|--------|
| listens | 10,000 | 934K | 46.5M |
| likes | 10,000 | — | 881K |
| dislikes | 10,000 | — | 108K |

### Schema

**Listens** (`listens.parquet`):
- `uid` — user ID
- `item_id` — track ID
- `timestamp` — time in 5-second bins (ordinal, usable for sorting)
- `is_organic` — 1 if user-initiated, 0 if recommendation-driven
- `played_ratio_pct` — playback completion percentage (0–100+, can exceed 100 on replay)
- `track_length_seconds` — track duration

**Likes / Dislikes** (`likes.parquet`, `dislikes.parquet`):
- `uid`, `item_id`, `timestamp`, `is_organic`

Dataset: https://huggingface.co/datasets/yandex/yambda

In [ ]:
from datasets import load_dataset

SIZE = '50m'
LOCAL = Path('yambda')

# Download via HuggingFace datasets library (correct path: flat/{size}/)
EVENT_FILES = ['listens', 'likes', 'dislikes']

for fname in EVENT_FILES:
    dest = LOCAL / f'flat/{SIZE}/{fname}.parquet'
    if not dest.exists():
        print(f'Downloading {fname}...')
        dest.parent.mkdir(parents=True, exist_ok=True)
        ds = load_dataset(
            'yandex/yambda',
            data_files={fname: f'flat/{SIZE}/{fname}.parquet'},
            split=fname,
        )
        ds.to_parquet(str(dest))
        print(f'  saved to {dest}')
    else:
        print(f'{fname}: already downloaded')

DATA = LOCAL / f'flat/{SIZE}'

listens  = pd.read_parquet(DATA / 'listens.parquet')
likes    = pd.read_parquet(DATA / 'likes.parquet')
dislikes = pd.read_parquet(DATA / 'dislikes.parquet')

# ── Normalize dtypes immediately after loading ────────────────────────────────
# Yambda stores uid and item_id as uint32. Mixed uint32/int64 comparisons and
# dict lookups can silently fail (map returns NaN, to_numpy(int32) corrupts the
# value). Cast everything to plain int32 once here so all downstream groupby,
# isin, and map operations use a single consistent dtype.
#
# played_ratio_pct is a nullable integer (pd.Int16 / pd.NA).
# Keeping it as a nullable integer causes problems in arithmetic and CSR
# construction. Convert to float32 now; pd.NA becomes NaN which we handle with
# fillna(0) for the weight matrix.
for df in [listens, likes, dislikes]:
    df['uid']     = df['uid'].astype(np.int32)
    df['item_id'] = df['item_id'].astype(np.int32)

listens['played_ratio_pct'] = (
    listens['played_ratio_pct']
    .astype('float32')   # pd.NA → NaN
    .fillna(0)           # NaN → 0.0  (no completion info = treat as 0%)
)
if 'track_length_seconds' in listens.columns:
    listens['track_length_seconds'] = (
        listens['track_length_seconds'].astype('float32').fillna(0)
    )

print(f'Listens  : {len(listens):>12,}  rows')
print(f'Likes    : {len(likes):>12,}  rows')
print(f'Dislikes : {len(dislikes):>12,}  rows')
print(f'\nUsers    : {listens.uid.nunique():>12,}')
print(f'Tracks   : {listens.item_id.nunique():>12,}')
print(f'\nDtypes after normalization:')
print(listens[['uid','item_id','played_ratio_pct']].dtypes.to_string())
listens.head()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Completion distribution
ratio = listens['played_ratio_pct'].clip(0, 120)
axes[0, 0].hist(ratio, bins=60, color='steelblue', edgecolor='white')
axes[0, 0].axvline(80, color='coral', linestyle='--', label='80% threshold')
axes[0, 0].set_title('Playback Completion (played_ratio_pct)')
axes[0, 0].set_xlabel('%')
axes[0, 0].legend()

# Organic vs recommended
org_counts = listens['is_organic'].value_counts()
axes[0, 1].bar(['Recommended', 'Organic'], org_counts.values, color=['coral', 'mediumseagreen'])
axes[0, 1].set_title('Organic vs Recommended Listens')
axes[0, 1].set_ylabel('Count')

# Listens per user (log)
user_listen_counts = listens.groupby('uid').size()
axes[0, 2].hist(user_listen_counts, bins=50, color='mediumpurple', edgecolor='white')
axes[0, 2].set_title('Listens per User')
axes[0, 2].set_yscale('log')

# Listens per track (log)
item_listen_counts = listens.groupby('item_id').size()
axes[1, 0].hist(item_listen_counts, bins=50, color='goldenrod', edgecolor='white')
axes[1, 0].set_title('Listens per Track')
axes[1, 0].set_yscale('log')

# Track length distribution
track_len = listens['track_length_seconds'].dropna().clip(0, 600)
axes[1, 1].hist(track_len, bins=60, color='darkcyan', edgecolor='white')
axes[1, 1].set_title('Track Length (seconds)')
axes[1, 1].axvline(210, color='coral', linestyle='--', label='3.5 min')
axes[1, 1].legend()

# Like / dislike ratio
total_listens = len(listens)
axes[1, 2].bar(
    ['Listens\n(all)', 'Likes', 'Dislikes'],
    [total_listens, len(likes), len(dislikes)],
    color=['steelblue', 'mediumseagreen', 'coral']
)
axes[1, 2].set_title('Event Type Volumes')
axes[1, 2].set_yscale('log')

plt.suptitle('Yambda-50M Dataset Exploration', fontsize=14)
plt.tight_layout()
plt.show()

completion_full = (listens['played_ratio_pct'] >= 80).mean()
like_rate_global = len(likes) / len(listens)
dislike_rate_global = len(dislikes) / len(listens)
print(f'Full completion rate (>=80%): {completion_full:.2%}')
print(f'Global like rate:             {like_rate_global:.3%}')
print(f'Global dislike rate:          {dislike_rate_global:.3%}')
print(f'Median listens per user:      {user_listen_counts.median():.0f}')
print(f'Median listens per track:     {item_listen_counts.median():.0f}')

### 1.1 Train / Test Split and Label Definition

We respect **temporal ordering** by splitting each user's listen history at the 80th percentile of their timestamps. Events before this point are *training data*; events after are *test data*.

### Graded Relevance from Multiple Events

Unlike MovieLens (1–5 star ratings), Yambda lets us build richer graded relevance from the combination of events:

| Grade | Condition | Interpretation |
|-------|-----------|----------------|
| **3** | User **liked** the track in test period | Strong explicit positive |
| **2** | User **completed** ≥ 80% of the track in test | Strong implicit positive |
| **1** | User **listened** (any completion) in test | Weak implicit positive |
| **0** | No interaction in test (candidate not listened to) | Unknown / implicit negative |

Tracks the user **disliked** are excluded from candidates (the system should never recommend them).

**Binary label**: `label_binary = 1` if grade ≥ 2 (completed listen or explicit like).

In [ ]:
# ── Temporal train/test split ─────────────────────────────────────────────────
# For each user: timestamps below their 80th percentile → train, above → test.
# We sort within each user by timestamp (works for both absolute and delta encoding).

listens_sorted = listens.sort_values(['uid', 'timestamp'])

train_mask = []
for uid, grp in tqdm(listens_sorted.groupby('uid', sort=False), desc='Splitting'):
    n = len(grp)
    cutoff = max(1, int(n * 0.8))
    flags = [True] * cutoff + [False] * (n - cutoff)
    train_mask.extend(flags)

train_mask = np.array(train_mask)
train_listens = listens_sorted[train_mask].copy()
test_listens  = listens_sorted[~train_mask].copy()

# Split likes/dislikes at the same timestamp threshold per user
user_cutoff_ts = train_listens.groupby('uid')['timestamp'].max().to_dict()

train_likes    = likes[likes.apply(lambda r: r.timestamp <= user_cutoff_ts.get(r.uid, 0), axis=1)]
test_likes     = likes[likes.apply(lambda r: r.timestamp >  user_cutoff_ts.get(r.uid, 0), axis=1)]
train_dislikes = dislikes[dislikes.apply(lambda r: r.timestamp <= user_cutoff_ts.get(r.uid, 0), axis=1)]

# Items the user disliked in training → never recommend
train_disliked = set(zip(train_dislikes.uid, train_dislikes.item_id))

print(f'Train listens : {len(train_listens):>10,}  |  users: {train_listens.uid.nunique():,}')
print(f'Test  listens : {len(test_listens):>10,}  |  users: {test_listens.uid.nunique():,}')
print(f'Train likes   : {len(train_likes):>10,}')
print(f'Test  likes   : {len(test_likes):>10,}  (ground truth for evaluation)')
print(f'Train dislikes: {len(train_dislikes):>10,}  (excluded from candidates)')

In [ ]:
# ── Build user / item indices ─────────────────────────────────────────────────

# Only users with test likes are interesting for evaluation
test_liked_users = set(test_likes['uid'].unique())

all_users = sorted(train_listens['uid'].unique())
all_items = sorted(train_listens['item_id'].unique())
user2idx  = {u: i for i, u in enumerate(all_users)}
item2idx  = {it: i for i, it in enumerate(all_items)}
idx2item  = {i: it for it, i in item2idx.items()}

eval_users = [u for u in test_liked_users if u in user2idx]

print(f'Users in index     : {len(all_users):,}')
print(f'Tracks in index    : {len(all_items):,}')
print(f'Users w/ test likes: {len(eval_users):,}  (used for evaluation)')

---
## Part 2: Candidate Generation

We generate candidates using two sources:

1. **ALS** — trained on completion-weighted listens. Instead of binary implicit feedback,
   we use `weight = clip(played_ratio_pct / 100, 0, 3)` as interaction strength.
   High-completion listens count more than abandoned ones.

2. **Popularity** — top globally-listened tracks not yet heard by the user.

Tracks the user already disliked are filtered from both sources.

In [ ]:
# ── Build completion-weighted user-item matrix ────────────────────────────────
# played_ratio_pct was normalized to float32 with fillna(0) at load time,
# so all arithmetic and CSR construction below is safe.

ui_agg = (
    train_listens
    .assign(weight=(train_listens['played_ratio_pct'].clip(0, 300) / 100).clip(upper=3))
    .groupby(['uid', 'item_id'])['weight']
    .sum()
    .clip(upper=10)
    .reset_index()
)

# Keep only users/items that are in our index
ui_agg = ui_agg[ui_agg['uid'].isin(user2idx) & ui_agg['item_id'].isin(item2idx)]
ui_agg = ui_agg[ui_agg['weight'] > 0]

row  = ui_agg['uid'].map(user2idx).to_numpy(dtype=np.int32)
col  = ui_agg['item_id'].map(item2idx).to_numpy(dtype=np.int32)
data = ui_agg['weight'].to_numpy(dtype=np.float32)

# Sanity check: all indices must be in bounds
assert row.max() < len(all_users), f"user index OOB: {row.max()} >= {len(all_users)}"
assert col.max() < len(all_items), f"item index OOB: {col.max()} >= {len(all_items)}"

user_item_csr = csr_matrix((data, (row, col)), shape=(len(all_users), len(all_items)))
user_item_csr.eliminate_zeros()

item_user_csr = user_item_csr.T.tocsr()
item_user_csr.eliminate_zeros()

print(f'User-item matrix shape: {user_item_csr.shape}')
print(f'Non-zero entries: {user_item_csr.nnz:,}')
print(f'Avg weight per entry: {data.mean():.3f}')
print(f'Weight dtype: {user_item_csr.data.dtype}')
print(f'Index dtype:  {user_item_csr.indices.dtype}')


In [ ]:
# ── Train ALS ────────────────────────────────────────────────────────────────

als_model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.1,
    iterations=20,
    random_state=42,
)
als_model.fit(item_user_csr)  # implicit expects (items × users)

print(f'User factors: {als_model.user_factors.shape}')
print(f'Item factors: {als_model.item_factors.shape}')

In [ ]:
# ── Generate candidates: ALS + popularity, filtered by dislikes ──────────────

N_ALS = 200
N_POP = 100

# Popularity: by total listen count in training
item_pop = train_listens.groupby('item_id').size().sort_values(ascending=False)
top_popular = item_pop.head(N_POP + 500).index.tolist()  # buffer

# Precompute per-user disliked sets
user_disliked_map = (
    train_dislikes.groupby('uid')['item_id']
    .apply(set)
    .to_dict()
)

# Precompute per-user heard sets (for filtering after recommend)
user_heard_map = (
    train_listens.groupby('uid')['item_id']
    .apply(set)
    .to_dict()
)

all_candidates = []

for uid in tqdm(eval_users, desc='Generating candidates'):
    uidx = user2idx[uid]

    als_ids, als_scores = als_model.recommend(
        uidx, user_item_csr[uidx], N=N_ALS + 300,
        filter_already_liked_items=False
    )

    user_heard    = user_heard_map.get(uid, set())
    user_disliked = user_disliked_map.get(uid, set())
    blocked       = user_heard | user_disliked

    # Build candidate pool — manually exclude heard and disliked items
    cands = {}
    for iid, score in zip([idx2item[i] for i in als_ids], als_scores):
        if iid not in blocked:
            cands[iid] = {'als_score': float(score), 'als_rank': len(cands) + 1}
        if len(cands) >= N_ALS:
            break

    # Add popularity candidates not yet seen or disliked
    pop_added = 0
    for iid in top_popular:
        if iid not in cands and iid not in blocked:
            cands[iid] = {'als_score': 0.0, 'als_rank': N_ALS + pop_added + 1}
            pop_added += 1
            if pop_added >= N_POP:
                break

    for iid, info in cands.items():
        all_candidates.append({
            'uid': uid, 'item_id': iid,
            'als_score': info['als_score'],
            'als_rank': info['als_rank'],
        })

candidates_df = pd.DataFrame(all_candidates)
print(f'Total candidates: {len(candidates_df):,}')
print(f'Avg per user:     {len(candidates_df) / len(eval_users):.1f}')


In [ ]:
# ── Attach graded labels ──────────────────────────────────────────────────────

# Like lookup (test period)
test_liked_set = set(zip(test_likes['uid'], test_likes['item_id']))

# Best completion per (user, item) in test
test_completion = (
    test_listens
    .groupby(['uid', 'item_id'])['played_ratio_pct']
    .max()
    .to_dict()
)

def graded_label(uid, item_id):
    if (uid, item_id) in test_liked_set:
        return 3
    comp = test_completion.get((uid, item_id), -1)
    if comp < 0:
        return 0          # not in test
    return 2 if comp >= 80 else 1

candidates_df['label_graded'] = [
    graded_label(r.uid, r.item_id)
    for r in tqdm(candidates_df.itertuples(), total=len(candidates_df), desc='Labels')
]
candidates_df['label_binary'] = (candidates_df['label_graded'] >= 2).astype(int)

# Recall: what fraction of test likes are covered by candidates?
test_like_pairs = set(zip(test_likes[test_likes.uid.isin(eval_users)].uid,
                          test_likes[test_likes.uid.isin(eval_users)].item_id))
cand_pairs = set(zip(candidates_df.uid, candidates_df.item_id))
recall = len(test_like_pairs & cand_pairs) / len(test_like_pairs)

print(f'Candidate recall (likes covered): {recall:.2%}')
print(f'\nLabel distribution:')
print(candidates_df.groupby('label_graded').size().rename({0:'0 (no interaction)', 1:'1 (partial listen)', 2:'2 (full listen)', 3:'3 (like)'}))

---
## Part 3: Feature Engineering

Yambda provides a fundamentally richer feature space than a ratings dataset:

| Feature group | Yambda-specific signals |
|---|---|
| **Item quality** | completion rate, like rate, dislike rate, net sentiment |
| **Item discovery** | organic listen fraction (quality of audience) |
| **User taste** | per-user like/dislike rate, typical completion, organic discovery rate |
| **User-Item history** | prior listen count, prior completion, liked before |
| **ALS** | personalized score, rank in candidates |

### Why completion rate matters

A track with 100K listens but average completion of 30% is a "trap" — users start it but abandon it. A track with 5K listens but 95% completion is a hidden gem. Pure popularity misses this entirely.

In [ ]:
# ── Item features ─────────────────────────────────────────────────────────────

item_listen_stats = train_listens.groupby('item_id').agg(
    item_n_listens        = ('uid', 'count'),
    item_n_unique_users   = ('uid', 'nunique'),
    item_avg_completion   = ('played_ratio_pct', 'mean'),
    item_med_completion   = ('played_ratio_pct', 'median'),
    item_full_listen_rate = ('played_ratio_pct', lambda x: (x >= 80).mean()),
    item_organic_rate     = ('is_organic', 'mean'),
    item_avg_length       = ('track_length_seconds', 'mean'),
).reset_index()

item_like_counts    = train_likes.groupby('item_id').size().rename('item_n_likes')
item_dislike_counts = train_dislikes.groupby('item_id').size().rename('item_n_dislikes')

item_features = (
    item_listen_stats
    .join(item_like_counts,    on='item_id', how='left')
    .join(item_dislike_counts, on='item_id', how='left')
)
item_features['item_n_likes']    = item_features['item_n_likes'].fillna(0)
item_features['item_n_dislikes'] = item_features['item_n_dislikes'].fillna(0)

# Derived quality signals
denom = item_features['item_n_listens'] + 1
item_features['item_like_rate']    = item_features['item_n_likes']    / denom
item_features['item_dislike_rate'] = item_features['item_n_dislikes'] / denom
item_features['item_net_sentiment']= item_features['item_like_rate'] - item_features['item_dislike_rate']
item_features['item_log_listens']  = np.log1p(item_features['item_n_listens'])
item_features['item_log_users']    = np.log1p(item_features['item_n_unique_users'])

item_feat_cols = [
    'item_log_listens', 'item_log_users',
    'item_avg_completion', 'item_med_completion', 'item_full_listen_rate',
    'item_organic_rate', 'item_avg_length',
    'item_like_rate', 'item_dislike_rate', 'item_net_sentiment',
]
print(f'Item features: {len(item_feat_cols)}')
item_features[item_feat_cols].describe().round(3)

In [ ]:
# ── User features ─────────────────────────────────────────────────────────────

user_listen_stats = train_listens.groupby('uid').agg(
    user_n_listens        = ('item_id', 'count'),
    user_n_unique_items   = ('item_id', 'nunique'),
    user_avg_completion   = ('played_ratio_pct', 'mean'),
    user_full_listen_rate = ('played_ratio_pct', lambda x: (x >= 80).mean()),
    user_organic_rate     = ('is_organic', 'mean'),
).reset_index()

user_like_counts    = train_likes.groupby('uid').size().rename('user_n_likes')
user_dislike_counts = train_dislikes.groupby('uid').size().rename('user_n_dislikes')

user_features = (
    user_listen_stats
    .join(user_like_counts,    on='uid', how='left')
    .join(user_dislike_counts, on='uid', how='left')
)
user_features['user_n_likes']    = user_features['user_n_likes'].fillna(0)
user_features['user_n_dislikes'] = user_features['user_n_dislikes'].fillna(0)

denom_u = user_features['user_n_listens'] + 1
user_features['user_like_rate']    = user_features['user_n_likes']    / denom_u
user_features['user_dislike_rate'] = user_features['user_n_dislikes'] / denom_u
user_features['user_log_listens']  = np.log1p(user_features['user_n_listens'])
user_features['user_log_items']    = np.log1p(user_features['user_n_unique_items'])

user_feat_cols = [
    'user_log_listens', 'user_log_items',
    'user_avg_completion', 'user_full_listen_rate',
    'user_organic_rate',
    'user_like_rate', 'user_dislike_rate',
]
print(f'User features: {len(user_feat_cols)}')
user_features[user_feat_cols].describe().round(3)

In [ ]:
# ── User-Item features (prior interaction history) ────────────────────────────
# For candidates that the user has already heard in training,
# we have explicit engagement signals.

ui_history = train_listens.groupby(['uid', 'item_id']).agg(
    ui_n_listens      = ('played_ratio_pct', 'count'),
    ui_avg_completion = ('played_ratio_pct', 'mean'),
    ui_max_completion = ('played_ratio_pct', 'max'),
).reset_index()

# Did the user like this item in training?
train_liked_set = set(zip(train_likes['uid'], train_likes['item_id']))
ui_history['ui_liked_in_train'] = [
    int((r.uid, r.item_id) in train_liked_set)
    for r in ui_history.itertuples()
]

ui_feat_cols = ['als_score', 'als_rank', 'ui_n_listens', 'ui_avg_completion', 'ui_max_completion', 'ui_liked_in_train']

print(f'User-Item features: {len(ui_feat_cols)}')

In [ ]:
# ── Merge all features into ranking dataset ────────────────────────────────────

feature_cols = item_feat_cols + user_feat_cols + ui_feat_cols

ranking_df = (
    candidates_df
    .merge(item_features[['item_id'] + item_feat_cols], on='item_id', how='left')
    .merge(user_features[['uid']    + user_feat_cols],  on='uid',     how='left')
    .merge(ui_history[['uid', 'item_id', 'ui_n_listens', 'ui_avg_completion',
                        'ui_max_completion', 'ui_liked_in_train']],
           on=['uid', 'item_id'], how='left')
)

ranking_df['ui_n_listens']      = ranking_df['ui_n_listens'].fillna(0)
ranking_df['ui_avg_completion'] = ranking_df['ui_avg_completion'].fillna(0)
ranking_df['ui_max_completion'] = ranking_df['ui_max_completion'].fillna(0)
ranking_df['ui_liked_in_train'] = ranking_df['ui_liked_in_train'].fillna(0)
ranking_df[feature_cols]        = ranking_df[feature_cols].fillna(0)

ranking_df = ranking_df.sort_values('uid').reset_index(drop=True)

print(f'Ranking dataset: {len(ranking_df):,} rows × {len(feature_cols)} features')
print(f'  Item features   : {len(item_feat_cols)}')
print(f'  User features   : {len(user_feat_cols)}')
print(f'  User-Item feats : {len(ui_feat_cols)}')
print(f'Positive rate (label>=2): {ranking_df.label_binary.mean():.3%}')
ranking_df.head()

In [ ]:
ranking_df.describe()

In [ ]:
# ── EDA: feature distributions by label ──────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

plot_feats = [
    ('als_score',            'ALS Score'),
    ('item_log_listens',     'log(Item Listens)'),
    ('item_avg_completion',  'Item Avg Completion %'),
    ('item_like_rate',       'Item Like Rate'),
    ('item_dislike_rate',    'Item Dislike Rate'),
    ('user_avg_completion',  'User Avg Completion %'),
    ('user_like_rate',       'User Like Rate'),
    ('ui_n_listens',         'Prior Listens (u,i)'),
]

for ax, (feat, title) in zip(axes.flat, plot_feats):
    data_by_grade = {
        g: ranking_df[ranking_df.label_graded == g][feat]
        for g in [0, 1, 2, 3]
    }
    colors = ['#aaaaaa', '#74c0fc', '#51cf66', '#ff6b6b']
    labels = ['0: no listen', '1: partial', '2: completed', '3: liked']
    lo = ranking_df[feat].quantile(0.01)
    hi = ranking_df[feat].quantile(0.99)
    for d, c, lbl in zip(data_by_grade.values(), colors, labels):
        ax.hist(d.clip(lo, hi), bins=40, alpha=0.5, label=lbl, color=c, density=True)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle('Feature Distributions by Relevance Grade', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
ranking_df = ranking_df.drop(
    columns=["ui_n_listens",
    "ui_avg_completion",
    "ui_max_completion",
    "ui_liked_in_train"]
)

In [ ]:
feature_cols = item_feat_cols + user_feat_cols + ['als_score', 'als_rank']

In [ ]:
# ── Train / evaluation split for the ranker ───────────────────────────────────
# User-level 80/20 split (separate from the temporal event split)

all_eval_users = ranking_df['uid'].unique()
rng = np.random.RandomState(42)
rng.shuffle(all_eval_users)
n_train_users = int(0.8 * len(all_eval_users))

ranker_train_users = set(all_eval_users[:n_train_users])
ranker_test_users  = set(all_eval_users[n_train_users:])

ranker_train = ranking_df[ranking_df.uid.isin(ranker_train_users)].copy()
ranker_test  = ranking_df[ranking_df.uid.isin(ranker_test_users)].copy()

X_train = ranker_train[feature_cols].values.astype(np.float32)
X_test  = ranker_test[feature_cols].values.astype(np.float32)

y_train_bin    = ranker_train['label_binary'].values
y_test_bin     = ranker_test['label_binary'].values
y_train_graded = ranker_train['label_graded'].values.astype(np.float32)
y_test_graded  = ranker_test['label_graded'].values.astype(np.float32)

groups_train = ranker_train['uid'].values
groups_test  = ranker_test['uid'].values

train_group_sizes = ranker_train.groupby('uid', sort=True).size().values
test_group_sizes  = ranker_test.groupby('uid', sort=True).size().values

print(f'Ranker train: {len(ranker_train):,} rows, {ranker_train.uid.nunique():,} users')
print(f'Ranker test : {len(ranker_test):,} rows, {ranker_test.uid.nunique():,} users')
print(f'Train positive rate: {y_train_bin.mean():.3%}')
print(f'Test  positive rate: {y_test_bin.mean():.3%}')

In [ ]:
ranker_train.head()

---
## Part 4: Ranking Loss Functions

### The Ranking Problem

Given a user $u$ and candidates $\mathcal{D}_u = \{d_1, \ldots, d_m\}$ with relevance labels $y_{u,d} \in \{0,1,2,3\}$, learn scoring function $f(u, d)$ such that items rank by decreasing relevance.

The ideal metric is **NDCG@K**:
$$\text{NDCG}@K = \frac{1}{\text{IDCG}@K} \sum_{k=1}^{K} \frac{2^{y_{(k)}} - 1}{\log_2(k+1)}$$

NDCG is non-differentiable (depends on sort order), so we use surrogate losses.

---

### 4.1 Pointwise Losses

Treat each (user, item) pair independently, ignoring the ranking structure:
$$\mathcal{L}_{\text{point}} = \sum_{(u,d)} \ell\bigl(f(u,d),\; y_{u,d}\bigr)$$

- **Logloss**: $\ell = -y\log\sigma(\hat{y}) - (1-y)\log(1-\sigma(\hat{y}))$ — binary labels
- **RMSE**: $\ell = (\hat{y} - y)^2$ — graded labels

**Problem**: A pair at rank 1 vs 2 gets the same gradient as rank 99 vs 100.

---

### 4.2 Pairwise Losses

For each query, compare all ordered pairs $(i, j)$ where $y_{ui} > y_{uj}$:
$$\mathcal{L}_{\text{pair}} = \sum_u \sum_{i \succ j} \ell\bigl(f(u,i) - f(u,j)\bigr)$$

#### PairLogit (BPR-style)
$$\ell(s) = \log(1 + e^{-s}), \quad s = f(u,i) - f(u,j)$$

Gradient for item $i$: $\frac{\partial \mathcal{L}}{\partial f_i} = \sum_{j \prec i} \sigma(f_j - f_i) - \sum_{j \succ i} \sigma(f_i - f_j)$

**Key property**: all pairs weighted equally — a pair at (rank 1, rank 2) contributes the same as (rank 50, rank 51).

#### YetiRank (CatBoost)

YetiRank bridges pairwise and listwise by sampling permutations from the **Plackett-Luce** model and weighting pairs by their expected NDCG contribution:

$$P(\pi) = \prod_{k=1}^{m} \frac{e^{f(u,\pi(k))}}{\sum_{j \geq k} e^{f(u,\pi(j))}}$$

For each sampled permutation, pairs at position $(p_i, p_j)$ are weighted by:
$$w_{ij} = \left|\frac{1}{\log_2(p_i+1)} - \frac{1}{\log_2(p_j+1)}\right| \cdot (2^{y_i} - 2^{y_j})$$

**Key difference from PairLogit:** pairs near the top of the list receive higher gradient signal, focusing the model on what matters most for NDCG@K.

With **graded labels** (0/1/2/3), YetiRank also uses the *exponential gain* $(2^{y_i} - 2^{y_j})$ which gives much higher weight to the like-vs-no-listen pair than to completed-vs-partial pairs — correctly reflecting NDCG semantics.

---

### 4.3 Listwise Losses — LambdaMART

LambdaRank defines pseudo-gradients that incorporate the exact metric change from swapping $i$ and $j$:
$$\lambda_{ij} = \frac{-1}{1 + e^{f_i - f_j}} \cdot |\Delta\text{NDCG}_{ij}|$$

where $|\Delta\text{NDCG}_{ij}|$ is computed using the **current model's ranking**. As training progresses, the pairs that are still wrong and would most hurt NDCG receive the largest gradients.

**LambdaMART** = LambdaRank + MART (gradient boosted trees). Implemented in LightGBM as `lambdarank` and in XGBoost as `rank:ndcg`.

---

### Summary

| Loss | Complexity | Pair weighting | NDCG alignment |
|------|-----------|----------------|----------------|
| Logloss | $O(N)$ | None | Low |
| PairLogit | $O(N^2)$ | Uniform | Medium |
| YetiRank | $O(N \log N)$ | NDCG × Plackett-Luce | High |
| LambdaMART | $O(N^2)$ | Exact $\lvert \Delta\text{NDCG} \rvert$ | Very high |

In [ ]:
# ── Evaluation metrics ────────────────────────────────────────────────────────

def ndcg_at_k(rels, k):
    rel = np.asarray(rels[:k], dtype=float)
    if rel.sum() == 0:
        return 0.0
    dcg  = (rel / np.log2(np.arange(2, len(rel) + 2))).sum()
    idcg = ((np.sort(rels)[::-1][:k].astype(float)) / np.log2(np.arange(2, k + 2))).sum()
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_ranking(df, score_col, label_col='label_binary', k=10):
    ndcgs, hrs, mrrs = [], [], []
    for uid, grp in df.groupby('uid'):
        grp_sorted = grp.sort_values(score_col, ascending=False)
        rels = grp_sorted[label_col].values
        ndcgs.append(ndcg_at_k(rels, k))
        hrs.append(int(rels[:k].sum() > 0))
        mrr = next((1.0/(r+1) for r, v in enumerate(rels[:k]) if v > 0), 0.0)
        mrrs.append(mrr)
    return {f'NDCG@{k}': np.mean(ndcgs), f'HR@{k}': np.mean(hrs),
            f'MRR@{k}': np.mean(mrrs), 'n_users': len(ndcgs)}


# Baseline: ALS score only
baseline = evaluate_ranking(ranker_test, 'als_score', k=10)
print('Baseline (ALS score only):')
for k, v in baseline.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

all_results = {'ALS (baseline)': baseline}

---
## Part 5: Training Rankers

### 5.1 Pointwise — CatBoost Logloss

Binary classifier predicting whether a user will complete or like a track (`label_binary = 1`). At inference, predicted probability is used as ranking score. No query structure is used during training.

In [ ]:
train_pool_pw = Pool(data=X_train, label=y_train_bin)
test_pool_pw  = Pool(data=X_test,  label=y_test_bin)

model_pw = CatBoostClassifier(
    loss_function='Logloss', eval_metric='AUC',
    iterations=500, learning_rate=0.05, depth=6,
    l2_leaf_reg=3.0, random_seed=42, verbose=100,
    early_stopping_rounds=50,
)
model_pw.fit(train_pool_pw, eval_set=test_pool_pw, use_best_model=True)

ranker_test['score_pw'] = model_pw.predict_proba(X_test)[:, 1]
metrics_pw = evaluate_ranking(ranker_test, 'score_pw', k=10)
print('\nPointwise (Logloss):')
for k, v in metrics_pw.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
all_results['Pointwise (Logloss)'] = metrics_pw

In [ ]:
fi = pd.Series(model_pw.feature_importances_, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
fi.tail(20).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('CatBoost Logloss — Top 20 Feature Importances\n(Yambda-specific signals highlighted)', fontsize=12)
ax.set_xlabel('Importance')

# Highlight Yambda-specific features
yambda_specific = {'item_avg_completion', 'item_like_rate', 'item_dislike_rate',
                   'item_net_sentiment', 'item_full_listen_rate', 'item_organic_rate',
                   'user_like_rate', 'user_dislike_rate', 'user_avg_completion',
                   'ui_avg_completion', 'ui_n_listens', 'ui_liked_in_train'}
for patch, feat in zip(ax.patches, fi.tail(20).index):
    if feat in yambda_specific:
        patch.set_facecolor('coral')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='steelblue', label='General'), Patch(color='coral', label='Yambda-specific')],
          fontsize=10)
plt.tight_layout()
plt.show()

### 5.2 Pairwise — CatBoost PairLogit

We provide `group_id=uid` to tell CatBoost about the query structure. CatBoost auto-generates all within-query pairs $(i, j)$ where `label_binary[i] > label_binary[j]` and minimizes the pairwise logistic loss. All pairs are weighted equally.

In [ ]:
train_pool_pair = Pool(data=X_train, label=y_train_bin,    group_id=groups_train)
test_pool_pair  = Pool(data=X_test,  label=y_test_bin,     group_id=groups_test)

model_pair = CatBoostRanker(
    loss_function='PairLogit', eval_metric='NDCG:top=10',
    iterations=500, learning_rate=0.05, depth=6,
    l2_leaf_reg=3.0, random_seed=42, verbose=100,
    early_stopping_rounds=50,
)
model_pair.fit(train_pool_pair, eval_set=test_pool_pair, use_best_model=True)

ranker_test['score_pair'] = model_pair.predict(X_test)
metrics_pair = evaluate_ranking(ranker_test, 'score_pair', k=10)
print('\nPairwise (PairLogit):')
for k, v in metrics_pair.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
all_results['Pairwise (PairLogit)'] = metrics_pair

### 5.3 Pairwise / Listwise — CatBoost YetiRank

YetiRank uses **graded labels** (0/1/2/3) to compute more meaningful pair weights. The exponential gain
$2^y - 1$ in NDCG means the pair (like=3, no-listen=0) carries $\frac{2^3-1}{2^0-1}$ times more weight than (partial-listen=1, no-listen=0):

$$2^3 - 2^0 = 7 \quad \text{vs} \quad 2^1 - 2^0 = 1$$

This automatically focuses the model on correctly ranking liked tracks above unlisten tracks — exactly what we care about.

In [ ]:
train_pool_yeti = Pool(data=X_train, label=y_train_graded, group_id=groups_train)
test_pool_yeti  = Pool(data=X_test,  label=y_test_graded,  group_id=groups_test)

model_yeti = CatBoostRanker(
    loss_function='YetiRank', eval_metric='NDCG:top=10',
    iterations=500, learning_rate=0.05, depth=6,
    l2_leaf_reg=3.0, random_seed=42, verbose=100,
    early_stopping_rounds=50,
)
model_yeti.fit(train_pool_yeti, eval_set=test_pool_yeti, use_best_model=True)

ranker_test['score_yeti'] = model_yeti.predict(X_test)
metrics_yeti = evaluate_ranking(ranker_test, 'score_yeti', k=10)
print('\nYetiRank:')
for k, v in metrics_yeti.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
all_results['YetiRank'] = metrics_yeti

In [ ]:
# ── PairLogit vs YetiRank: learning curves ────────────────────────────────────

pair_evals = model_pair.get_evals_result()
yeti_evals = model_yeti.get_evals_result()

fig, ax = plt.subplots(figsize=(10, 5))

for evals, name, color in [
    (pair_evals, 'PairLogit', 'steelblue'),
    (yeti_evals, 'YetiRank',  'coral'),
]:
    for split, linestyle, alpha in [('learn', '-', 0.4), ('validation', '--', 1.0)]:
        if split not in evals or not evals[split]:
            continue
        key = list(evals[split].keys())[0]
        ax.plot(evals[split][key], color=color, linestyle=linestyle,
                alpha=alpha, label=f'{name} {split}')

ax.set_title('NDCG@10 during training: PairLogit vs YetiRank')
ax.set_xlabel('Iteration')
ax.legend()
plt.tight_layout()
plt.show()

### 5.4 Listwise — CatBoost QuerySoftMax

CatBoost provides a **listwise** loss called `QuerySoftMax` — a query-level cross-entropy that treats each query as a multi-class softmax over all its items:

$$\mathcal{L}_{\text{softmax}} = -\sum_u \sum_{i:\,y_{ui}>0} y_{ui} \cdot \log \frac{e^{f(u,i)}}{\sum_{j \in \mathcal{D}_u} e^{f(u,j)}}$$

**Key differences from YetiRank:**
- YetiRank: pair-based, NDCG-weighted, requires Plackett-Luce sampling
- QuerySoftMax: list-based, proportional to relevance weight, $O(N)$ per query

QuerySoftMax is faster to train but uses a softer alignment with NDCG. It is often competitive with YetiRank when positives are very sparse.

### All CatBoost ranking losses — recap

| Loss | Paradigm | Label type | Pair weighting |
|------|----------|------------|----------------|
| `Logloss` | Pointwise | Binary | None |
| `PairLogit` | Pairwise | Binary or graded | Uniform |
| `YetiRank` | Pairwise → Listwise | Graded | NDCG × Plackett-Luce |
| `QuerySoftMax` | Listwise | Graded | Relevance-proportional |


In [ ]:
# ── CatBoost QuerySoftMax ────────────────────────────────────────────────────

train_pool_qs = Pool(data=X_train, label=y_train_graded, group_id=groups_train)
test_pool_qs  = Pool(data=X_test,  label=y_test_graded,  group_id=groups_test)

model_qs = CatBoostRanker(
    loss_function='QuerySoftMax', eval_metric='NDCG:top=10',
    iterations=500, learning_rate=0.05, depth=6,
    l2_leaf_reg=3.0, random_seed=42, verbose=100,
    early_stopping_rounds=50,
)
model_qs.fit(train_pool_qs, eval_set=test_pool_qs, use_best_model=True)

ranker_test['score_qs'] = model_qs.predict(X_test)
metrics_qs = evaluate_ranking(ranker_test, 'score_qs', k=10)
print('\nQuerySoftMax:')
for k, v in metrics_qs.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
all_results['QuerySoftMax'] = metrics_qs

# ── All four CatBoost losses side-by-side ────────────────────────────────────
catboost_results = {
    'Logloss\n(pointwise)': all_results['Pointwise (Logloss)'],
    'PairLogit\n(pairwise)': all_results['Pairwise (PairLogit)'],
    'YetiRank\n(pair→list)': all_results['YetiRank'],
    'QuerySoftMax\n(listwise)': metrics_qs,
}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cb_labels = list(catboost_results.keys())
cb_colors = ['#74b9ff', '#55efc4', '#fd79a8', '#fdcb6e']

for ax, metric in zip(axes, ['NDCG@10', 'HR@10', 'MRR@10']):
    vals = [catboost_results[s][metric] for s in cb_labels]
    bars = ax.bar(range(len(cb_labels)), vals, color=cb_colors)
    ax.set_xticks(range(len(cb_labels)))
    ax.set_xticklabels(cb_labels, fontsize=9)
    ax.set_title(metric, fontsize=13)
    ax.set_ylim(min(vals) * 0.95, max(vals) * 1.05)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('CatBoost: All Four Ranking Losses Compared', fontsize=13)
plt.tight_layout()
plt.show()


### 5.5 Comparison with Other Libraries — LightGBM & XGBoost

CatBoost is our primary tool in this seminar. For completeness, we also run **LightGBM LambdaRank** and **XGBoost rank:ndcg** — both implement LambdaMART but with different API conventions.

| | CatBoost | LightGBM | XGBoost |
|---|---|---|---|
| Group spec | `group_id=` per row | `group=` size array | `dmatrix.set_group()` |
| Graded gain | automatic from label | `label_gain=[...]` | automatic |
| GPU support | Yes | Yes | Yes |
| Native categoricals | Yes | No | No |
| Listwise losses | YetiRank, QuerySoftMax | lambdarank, rank_xendcg | rank:ndcg, rank:map |


In [ ]:
# ── LightGBM LambdaRank ───────────────────────────────────────────────────────

lgb_train = lgb.Dataset(
    X_train, label=y_train_graded.astype(int),
    group=train_group_sizes, free_raw_data=False,
)
lgb_test = lgb.Dataset(
    X_test, label=y_test_graded.astype(int),
    group=test_group_sizes, reference=lgb_train, free_raw_data=False,
)

model_lgb = lgb.train(
    {
        'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10],
        'label_gain': [0, 1, 3, 7],   # 2^y - 1 for y in {0,1,2,3}
        'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 6,
        'min_data_in_leaf': 10, 'n_jobs': -1, 'seed': 42, 'verbosity': -1,
    },
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_test],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100),
    ],
)
ranker_test['score_lgb'] = model_lgb.predict(X_test)
metrics_lgb = evaluate_ranking(ranker_test, 'score_lgb', k=10)
print('LightGBM LambdaRank:')
for k, v in metrics_lgb.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
all_results['LightGBM (LambdaRank)'] = metrics_lgb

# ── XGBoost rank:ndcg ──────────────────────────────────────────────────────────

xgb_train = xgb.DMatrix(X_train, label=y_train_graded)
xgb_train.set_group(train_group_sizes)
xgb_test  = xgb.DMatrix(X_test,  label=y_test_graded)
xgb_test.set_group(test_group_sizes)

evals_result = {}
model_xgb = xgb.train(
    {
        'objective': 'rank:ndcg', 'eval_metric': 'ndcg@10',
        'eta': 0.05, 'max_depth': 6, 'min_child_weight': 5,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'seed': 42, 'verbosity': 0,
    },
    xgb_train,
    num_boost_round=500,
    evals=[(xgb_test, 'eval')],
    evals_result=evals_result,
    early_stopping_rounds=50,
    verbose_eval=100,
)
ranker_test['score_xgb'] = model_xgb.predict(xgb_test)
metrics_xgb = evaluate_ranking(ranker_test, 'score_xgb', k=10)
print('\nXGBoost rank:ndcg:')
for k, v in metrics_xgb.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
all_results['XGBoost (rank:ndcg)'] = metrics_xgb


---
## Part 6: Negative Sampling Strategies

### The Sampling Problem

In Yambda-50M, a typical user has listened to ~4,650 tracks. Our candidate set has ~300 items per user. Of these, only ~2–5% are positives (completed listens or likes). The rest are negatives — but not all negatives are equal:

```
All candidate negatives
├── Easy negatives:  unpopular tracks, very dissimilar to user's taste
│   → Model already scores these low. Gradient contribution ≈ 0.
│   → Waste of compute, no learning signal.
│
└── Hard negatives:  popular tracks, similar to user's taste, but not interacted
    → Model currently confused about these.
    → Large gradient, most informative pairs.
```

### Four Strategies

#### 1. All negatives (no sampling)
Use every candidate with `label_binary = 0`. Highly imbalanced (~1:25).

#### 2. Random negatives (1:ratio)
Sample uniformly from negatives. Unbiased but many easy examples.
$$P(\text{sample} \mid \text{negative}) = \text{const}$$

#### 3. Hard negatives (ALS-based)
Select negatives with the **highest ALS score** — tracks the retrieval model thinks the user would like, but they haven't interacted with yet (or it was a recommendation miss).
$$\text{Hard negatives} = \operatorname{top-}K \text{ by } \texttt{als\_score} \cap \{\text{label\_binary}=0\}$$

**Risk**: Some hard negatives may be *false* hard negatives — tracks the user would have enjoyed but never had the chance to hear.

#### 4. Mixed (50% hard + 50% random)
Balance informativeness with diversity. Standard in production systems.

### Positive:Negative Ratio

For pointwise models the ratio shifts the decision boundary:
- Too few negatives → model predicts positive too liberally (low precision)
- Too many negatives → model predicts positive too conservatively (low recall)

In [ ]:
def sample_negatives(train_df, strategy='random', neg_ratio=5, random_state=42):
    """
    strategy: 'all' | 'random' | 'hard' | 'mixed'
    hard negatives = highest als_score negatives (tracks ALS thinks user would like)
    """
    rng = np.random.RandomState(random_state)
    parts = []

    for uid, grp in train_df.groupby('uid'):
        pos = grp[grp.label_binary == 1]
        neg = grp[grp.label_binary == 0]
        n_need = len(pos) * neg_ratio

        if strategy == 'all':
            parts.append(grp)
            continue
        if len(neg) == 0:
            parts.append(pos)
            continue

        if strategy == 'random':
            sampled = neg.sample(n=min(n_need, len(neg)), random_state=rng.randint(1e6))

        elif strategy == 'hard':
            sampled = neg.nlargest(min(n_need, len(neg)), 'als_score')

        elif strategy == 'mixed':
            n_hard = min(n_need // 2, len(neg))
            hard   = neg.nlargest(n_hard, 'als_score')
            rest   = neg[~neg.index.isin(hard.index)]
            n_rand = min(n_need - n_hard, len(rest))
            rand   = rest.sample(n=n_rand, random_state=rng.randint(1e6)) if n_rand > 0 else pd.DataFrame()
            sampled = pd.concat([hard, rand])

        parts.append(pd.concat([pos, sampled]))

    return pd.concat(parts).sort_values('uid').reset_index(drop=True)


strategies = {
    'all':    sample_negatives(ranker_train, strategy='all'),
    'random': sample_negatives(ranker_train, strategy='random', neg_ratio=5),
    'hard':   sample_negatives(ranker_train, strategy='hard',   neg_ratio=5),
    'mixed':  sample_negatives(ranker_train, strategy='mixed',  neg_ratio=5),
}

for name, df in strategies.items():
    n_pos = df.label_binary.sum()
    n_neg = (df.label_binary == 0).sum()
    print(f'{name:8s}: {len(df):7,} rows | {n_pos:5,} pos | {n_neg:6,} neg | ratio 1:{n_neg/max(n_pos,1):.1f}')

In [ ]:
sampling_results = {}

for strat_name, train_sample in strategies.items():
    X_tr = train_sample[feature_cols].values.astype(np.float32)
    y_tr = train_sample['label_binary'].values

    m = CatBoostClassifier(
        loss_function='Logloss', iterations=400,
        learning_rate=0.05, depth=6,
        random_seed=42, verbose=0,
    )
    m.fit(Pool(data=X_tr, label=y_tr))

    ranker_test[f'score_s_{strat_name}'] = m.predict_proba(X_test)[:, 1]
    metrics = evaluate_ranking(ranker_test, f'score_s_{strat_name}', k=10)
    sampling_results[strat_name] = metrics
    print(f'  {strat_name:8s} → NDCG@10={metrics["NDCG@10"]:.4f}, HR@10={metrics["HR@10"]:.4f}, MRR@10={metrics["MRR@10"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
strat_labels = list(sampling_results.keys())
colors = ['#4878CF', '#6ACC65', '#D65F5F', '#B47CC7']

for ax, metric in zip(axes, ['NDCG@10', 'HR@10', 'MRR@10']):
    vals = [sampling_results[s][metric] for s in strat_labels]
    bars = ax.bar(strat_labels, vals, color=colors)
    ax.set_title(metric, fontsize=13)
    ax.set_ylim(min(vals) * 0.95, max(vals) * 1.05)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Negative Sampling Strategy Comparison (CatBoost Logloss, Yambda-50M)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Effect of positive:negative ratio (mixed strategy) ─────────────────────────

ratio_results = {}
for neg_ratio in [1, 3, 5, 10, 20]:
    sample = sample_negatives(ranker_train, strategy='mixed', neg_ratio=neg_ratio)
    X_tr = sample[feature_cols].values.astype(np.float32)
    y_tr = sample['label_binary'].values
    m = CatBoostClassifier(
        loss_function='Logloss', iterations=400,
        learning_rate=0.05, depth=6,
        random_seed=42, verbose=0,
    )
    m.fit(Pool(data=X_tr, label=y_tr))
    ranker_test[f'score_r{neg_ratio}'] = m.predict_proba(X_test)[:, 1]
    ratio_results[f'1:{neg_ratio}'] = evaluate_ranking(ranker_test, f'score_r{neg_ratio}', k=10)
    print(f'  1:{neg_ratio:2d} → NDCG@10={ratio_results[f"1:{neg_ratio}"]["NDCG@10"]:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(ratio_results.keys()), [v['NDCG@10'] for v in ratio_results.values()],
        marker='o', label='NDCG@10', color='steelblue')
ax.plot(list(ratio_results.keys()), [v['HR@10'] for v in ratio_results.values()],
        marker='s', label='HR@10', color='coral')
ax.set_title('Effect of Positive:Negative Ratio (Mixed Sampling, CatBoost Logloss)')
ax.set_xlabel('Train Ratio (positives:negatives)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 7: Final Comparison

In [ ]:
comparison = pd.DataFrame(all_results).T[['NDCG@10', 'HR@10', 'MRR@10']]

print('=' * 62)
print(f'{"Model":30s} | {"NDCG@10":>8s} | {"HR@10":>8s} | {"MRR@10":>8s}')
print('-' * 62)
for model, row in comparison.iterrows():
    print(f'{model:30s} | {row["NDCG@10"]:>8.4f} | {row["HR@10"]:>8.4f} | {row["MRR@10"]:>8.4f}')
print('=' * 62)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
palette = plt.cm.Set2(np.linspace(0, 0.8, len(comparison)))

for ax, metric in zip(axes, ['NDCG@10', 'HR@10', 'MRR@10']):
    vals = comparison[metric].values
    bars = ax.barh(comparison.index, vals, color=palette)
    ax.set_title(metric, fontsize=13)
    ax.set_xlim(0, max(vals) * 1.15)
    for bar, v in zip(bars, vals):
        ax.text(v + 0.001, bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=9)

plt.suptitle('Ranking Model Comparison — Yambda-50M', fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 8: Ablation — Value of Multi-Event Features

A natural question: **how much do likes, dislikes, and completion rate actually help?** Could we achieve similar ranking quality using only listen counts?

We train YetiRank with four progressively richer feature sets:

| Feature set | Features included |
|-------------|-------------------|
| **ALS only** | `als_score`, `als_rank` — no engineered features |
| **+ Listen counts** | ALS + item/user listen counts, `ui_n_listens` |
| **+ Completion** | All above + `played_ratio_pct`-derived features |
| **+ Likes/Dislikes** | All features (full set from Part 3) |

This directly quantifies the marginal value each Yambda event type adds to ranking quality.

In [ ]:
# ── Feature set ablation ──────────────────────────────────────────────────────

feature_sets = {
    'ALS only': [
        'als_score', 'als_rank',
    ],
    '+ Listen counts': [
        'als_score', 'als_rank',
        'item_log_listens', 'item_log_users',
        'user_log_listens', 'user_log_items',
        # 'ui_n_listens',
    ],
    '+ Completion': [
        'als_score', 'als_rank',
        'item_log_listens', 'item_log_users',
        'item_avg_completion', 'item_med_completion', 'item_full_listen_rate',
        'item_avg_length', 'item_organic_rate',
        'user_log_listens', 'user_log_items',
        'user_avg_completion', 'user_full_listen_rate', 'user_organic_rate',
        # 'ui_n_listens', 'ui_avg_completion', 'ui_max_completion',
    ],
    '+ Likes/Dislikes': feature_cols,  # full feature set
}

ablation_results = {}
train_sample_abl = sample_negatives(ranker_train, strategy='mixed', neg_ratio=5)

for feat_name, feats in feature_sets.items():
    X_tr = train_sample_abl[feats].values.astype(np.float32)
    X_te = ranker_test[feats].values.astype(np.float32)

    pool_tr = Pool(
        data=X_tr,
        label=train_sample_abl['label_graded'].values.astype(np.float32),
        group_id=train_sample_abl['uid'].values,
    )

    m = CatBoostRanker(
        loss_function='YetiRank', eval_metric='NDCG:top=10',
        iterations=400, learning_rate=0.05, depth=6,
        random_seed=42, verbose=0,
    )
    m.fit(pool_tr)

    score_col = f'abl_{feat_name}'
    ranker_test[score_col] = m.predict(X_te)
    ablation_results[feat_name] = evaluate_ranking(ranker_test, score_col, k=10)
    ndcg = ablation_results[feat_name]['NDCG@10']
    print(f'  {feat_name:22s} → NDCG@10={ndcg:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
feat_names = list(ablation_results.keys())
palette_blues = plt.cm.Blues(np.linspace(0.35, 0.85, len(feat_names)))

for ax, metric in zip(axes, ['NDCG@10', 'HR@10', 'MRR@10']):
    vals = [ablation_results[s][metric] for s in feat_names]
    bars = ax.bar(range(len(feat_names)), vals, color=palette_blues)
    ax.set_xticks(range(len(feat_names)))
    ax.set_xticklabels(feat_names, rotation=20, ha='right', fontsize=9)
    ax.set_title(metric, fontsize=13)
    ax.set_ylim(min(vals) * 0.92, max(vals) * 1.05)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Ablation: Marginal Value of Each Event Type (YetiRank)', fontsize=13)
plt.tight_layout()
plt.show()

# Marginal gains
print('Marginal NDCG@10 gains:')
prev = None
for name in feat_names:
    cur = ablation_results[name]['NDCG@10']
    gain = f' (+{cur - prev:.4f})' if prev is not None else ''
    print(f'  {name:22s}: {cur:.4f}{gain}')
    prev = cur


---
## Part 9: The `is_organic` Signal

Yambda records **how** a user found each track:
- `is_organic = 1` — user-initiated (search, artist page, self-built playlist)
- `is_organic = 0` — recommendation-driven (algorithmic exposure)

This is unusual for open datasets and has two uses in ranking:

**Item quality**: A track with high `item_organic_rate` is sought out actively — it has intrinsic appeal beyond algorithmic push. A track with low organic rate might be popular purely because the algorithm keeps inserting it.

**User reliability**: A user with high `user_organic_rate` is self-directed. Their implicit signals (completions, listens) are more reliable because they chose the content, not the algorithm.

**Confounding risk**: We need to check whether completion rates actually differ between organic and recommended contexts — if they don't, the signal is redundant.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Correlation: item popularity vs organic rate
ax = axes[0]
sample_items = item_features.sample(min(5000, len(item_features)), random_state=42)
ax.scatter(
    sample_items['item_log_listens'], sample_items['item_organic_rate'],
    alpha=0.3, s=10, color='steelblue',
)
ax.set_xlabel('log(item listen count)')
ax.set_ylabel('Organic listen rate')
ax.set_title('Popularity vs Organic Rate')
corr = sample_items[['item_log_listens', 'item_organic_rate']].corr().iloc[0, 1]
ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes, va='top', fontsize=11)

# 2. Completion: organic vs recommended listens
ax = axes[1]
org_comp = train_listens[train_listens.is_organic == 1]['played_ratio_pct'].clip(0, 120)
rec_comp = train_listens[train_listens.is_organic == 0]['played_ratio_pct'].clip(0, 120)
ax.hist(org_comp, bins=50, alpha=0.6, label=f'Organic (n={len(org_comp):,})',
        color='mediumseagreen', density=True)
ax.hist(rec_comp, bins=50, alpha=0.6, label=f'Recommended (n={len(rec_comp):,})',
        color='coral', density=True)
ax.set_title('Completion: Organic vs Recommended')
ax.set_xlabel('played_ratio_pct')
ax.legend(fontsize=9)
print(f'Organic    avg completion: {org_comp.mean():.1f}%')
print(f'Recommended avg completion: {rec_comp.mean():.1f}%')

# 3. Item organic rate by relevance grade in ranking set
ax = axes[2]
for grade, color, lbl in [
    (0, '#aaaaaa', '0: no interaction'),
    (2, '#51cf66', '2: completed (>=80%)'),
    (3, '#ff6b6b', '3: liked'),
]:
    sub = ranking_df[ranking_df.label_graded == grade]['item_organic_rate']
    ax.hist(sub, bins=40, alpha=0.5, label=lbl, density=True)
ax.set_title('Item Organic Rate by Relevance Grade')
ax.set_xlabel('item_organic_rate')
ax.legend(fontsize=9)

plt.suptitle('The is_organic Signal in Yambda', fontsize=13)
plt.tight_layout()
plt.show()

# Takeaway: do liked/completed tracks have higher organic rate?
print('\nMean item_organic_rate by grade:')
print(ranking_df.groupby('label_graded')['item_organic_rate'].mean().rename(
    {0: '0 (no listen)', 1: '1 (partial)', 2: '2 (completed)', 3: '3 (liked)'}
).to_string())


---
## Exercises

### Exercise 1 — Alternative Graded Label  *(Easy)*

The current scheme has 4 grades (0–3). Add a **grade 4** for tracks the user both **liked AND completed ≥ 90%** in the test period (a "super-positive" signal reflecting maximum engagement).

1. Modify `graded_label()` to output grades 0–4
2. Retrain YetiRank (no other changes)
3. Update `label_gain=[0, 1, 3, 7, 15]` in LightGBM and retrain LambdaRank
4. Compare NDCG@10 before/after

*Why might a 5-grade scheme help? When might it hurt (hint: data sparsity)?*

---

### Exercise 2 — Organic-Only ALS Score  *(Medium)*

Train a second ALS model on **organic listens only** (`is_organic == 1` in the listen matrix). The intuition: this model captures genuine user taste uncontaminated by algorithmic exposure bias.

1. Filter `train_listens` to organic rows before building the CSR matrix
2. Train ALS with the same hyperparameters
3. Add `organic_als_score` and `organic_als_rank` as new features
4. Retrain YetiRank with all features + the two new ones
5. Check whether `organic_als_score` appears in the top-10 feature importance list

---

### Exercise 3 — PairLogit with Graded Labels  *(Medium)*

CatBoost PairLogit with graded labels generates more pairs: $(i, j)$ whenever `label[i] > label[j]`, so a pair `(liked=3, completed=2)` also trains the model.

1. Retrain PairLogit using `y_train_graded` instead of `y_train_bin`
2. Compare NDCG@10 vs binary PairLogit and vs YetiRank with graded labels
3. Explain: *why does YetiRank benefit more from graded labels than PairLogit does?*

*(Hint: PairLogit weights all pairs equally; YetiRank additionally scales by $2^{y_i} - 2^{y_j}$, which is 7× larger for a like-vs-no-listen pair than for a partial-vs-no-listen pair.)*

---

### Exercise 4 — Curriculum Negative Sampling  *(Hard)*

**Curriculum learning**: start easy, get harder as the model improves.

1. Train CatBoost Logloss for 200 iterations with `strategy='random', neg_ratio=5`
2. Score all training negatives with this model
3. Keep only negatives where `predicted_proba > 0.3` (the model is still confused)
4. Continue training for another 200 iterations starting from the saved model
5. Compare final NDCG@10 to static hard / mixed / random strategies

*CatBoost tip: pass `init_model=model` to `.fit()` to warm-start from a saved checkpoint.*

---

### Exercise 5 — Rank-Normalized Score Fusion  *(Easy)*

Instead of picking the single best model, combine them. Raw scores are on different scales, so normalize to per-user percentile ranks first:

```python
def rank_normalize(df, col, uid_col='uid'):
    df[col + '_rn'] = df.groupby(uid_col)[col].rank(pct=True)
    return df

for col in ['score_yeti', 'score_lgb', 'score_xgb']:
    rank_normalize(ranker_test, col)

ranker_test['score_ensemble'] = (
    0.4 * ranker_test['score_yeti_rn'] +
    0.4 * ranker_test['score_lgb_rn']  +
    0.2 * ranker_test['score_xgb_rn']
)
```

Grid-search the weights $(w_1, w_2, w_3)$ with $w_1 + w_2 + w_3 = 1$ using the validation set. By how much does the best ensemble beat the single best model?

---
## Conclusions & Key Takeaways

### Why Yambda is better than MovieLens for this seminar

| Aspect | MovieLens | Yambda |
|--------|-----------|--------|
| Signals | 1 (rating 1–5) | 5 (listen, completion, like, dislike, organic) |
| Label richness | Ordinal 1–5 | Graded 0–3 built from independent event types |
| Item quality proxy | avg rating only | completion rate + like rate + dislike rate |
| User taste proxy | avg rating given | completion profile + like/dislike ratio |
| User-item history | single rating | listen count + avg completion + liked before |
| `is_organic` flag | No | Yes — captures discovery mode |

### CatBoost ranking losses — full picture

| Loss | Paradigm | Label | Pair weighting | Best for |
|------|----------|-------|----------------|----------|
| `Logloss` | Pointwise | Binary | None | Calibrated scores, simple baseline |
| `PairLogit` | Pairwise | Binary/graded | Uniform | All pairs matter equally |
| `YetiRank` | Pairwise→Listwise | Graded | NDCG × Plackett-Luce | Direct NDCG optimization |
| `QuerySoftMax` | Listwise | Graded | Relevance-proportional | Sparse positives, fast training |

### PairLogit vs YetiRank — the central contrast

```
PairLogit: gradient ∝ σ(f_j − f_i)
           All pairs weighted equally.
           Does not know which pairs affect NDCG@K most.

YetiRank:  gradient ∝ σ(f_j − f_i) × |ΔNDCG_ij| × (2^y_i − 2^y_j)
           Pairs near the top of the list get higher gradient.
           Graded labels give 7× more signal for like-vs-no-listen than partial-vs-no-listen.
```

### Negative sampling

- **Hard negatives** (highest ALS score among non-interacted) → most informative, but risk false negatives
- **Mixed 50/50** (hard + random) → best of both worlds; standard in production
- **Optimal ratio** ≈ 1:5; grid search on validation when in doubt

### Feature ablation takeaways

- ALS score alone is a strong baseline — collaborative filtering captures most of the signal
- Completion rate features add a meaningful boost — they distinguish quality from quantity
- Like/dislike rates add the final increment — explicit signals are rare but very informative
- `is_organic` is a useful item quality indicator; organic listens complete more often

### Library API cheat sheet

| Library | Groups | Ranking losses |
|---------|--------|----------------|
| **CatBoost** | `Pool(..., group_id=uid_per_row)` | Logloss, PairLogit, YetiRank, QuerySoftMax |
| **LightGBM** | `Dataset(..., group=size_array)` | lambdarank, rank_xendcg |
| **XGBoost** | `dmatrix.set_group(size_array)` | rank:ndcg, rank:pairwise, rank:map |

### Further Reading

- Burges et al. (2005) — [Learning to rank using gradient descent (RankNet)](https://icml.cc/2015/wp-content/uploads/2015/06/icml_ranking.pdf)
- Burges (2010) — [From RankNet to LambdaRank to LambdaMART: An Overview](https://www.microsoft.com/en-us/research/publication/from-ranknet-to-lambdarank-to-lambdamart-an-overview/)
- Gulin et al. (2011) — [A Novel Approximation Framework for Scoring and Ranking (YetiRank)](https://arxiv.org/abs/1106.0722)
- Yandex (2025) — [Yambda: A Large-Scale Music Recommendation Dataset](https://arxiv.org/abs/2505.22238)
